# NLP-20: Huấn Luyện & Đánh Giá Mô Hình Transformer Seq2Seq (BARTpho)
### Nhóm 10 - Khoa Công nghệ Thông tin - UNETI

Notebook này hướng dẫn quy trình Fine-tuning và đánh giá mô hình BARTpho (`vinai/bartpho-syllable`):
- Tải checkpoint Seq2Seq Transformer từ HuggingFace Hub.
- Tiền xử lý văn bản tiếng Việt và ánh xạ tokens với attention masks.
- Cấu hình siêu tham số: Beam search, length penalty, no_repeat_ngram_size.
- Đánh giá các chỉ số ROUGE-1/2/L và Factuality Score (kiểm soát ảo giác số liệu/thực thể).

In [ ]:
import os
import sys
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.utils import load_jsonl
from src.baselines.lead_k import LeadKSummarizer
from src.baselines.tfidf_summarizer import TFIDFSummarizer
from src.models.bartpho_summarizer import BARTphoSummarizer
from src.evaluation.rouge_evaluator import ROUGEEvaluator
from src.evaluation.fact_checker import FactualityChecker

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Thiết bị tính toán khả dụng: {device.upper()}")

## 1. Nạp tập kiểm thử Test Set và Khởi tạo Mô hình

In [ ]:
test_path = os.path.join(PROJECT_ROOT, "data", "processed", "test.jsonl")
test_data = load_jsonl(test_path)
print(f"Nạp thành công {len(test_data)} mẫu test.")

bartpho = BARTphoSummarizer(lazy_load=True)
evaluator = ROUGEEvaluator()
print("Mô hình và bộ đánh giá ROUGE đã sẵn sàng.")

## 2. Thực hiện Dự đoán và Đánh giá ROUGE

In [ ]:
docs = [s["document"] for s in test_data]
refs = [s["summary"] for s in test_data]

preds = [bartpho.summarize(d)["summary"] for d in docs]
scores = evaluator.evaluate_corpus(preds, refs)

print("=== KẾT QUẢ ROUGE TRÊN TẬP TEST ===")
print(f"ROUGE-1: Precision={scores['rouge-1']['precision']:.4f} | Recall={scores['rouge-1']['recall']:.4f} | F1={scores['rouge-1']['f1']:.4f}")
print(f"ROUGE-2: Precision={scores['rouge-2']['precision']:.4f} | Recall={scores['rouge-2']['recall']:.4f} | F1={scores['rouge-2']['f1']:.4f}")
print(f"ROUGE-L: Precision={scores['rouge-l']['precision']:.4f} | Recall={scores['rouge-l']['recall']:.4f} | F1={scores['rouge-l']['f1']:.4f}")